In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# пойдем по пути по которому ходят коты и токи
TITANIC_FILE = "data/Titanic-Dataset.csv"
AB_SESSIONS_FILE = "data/sessions_data.csv"
AB_USERS_FILE = "data/users_data.csv"

df_titanic = pd.read_csv(TITANIC_FILE)
df_sessions = pd.read_csv(AB_SESSIONS_FILE)
df_users = pd.read_csv(AB_USERS_FILE)

df_ab = df_sessions.merge(df_users, on='user_id', how='left')
df_ab['booked'] = df_ab['booking_timestamp'].notna().astype(int)
df_ab['session_start_timestamp'] = pd.to_datetime(df_ab['session_start_timestamp'])
df_ab['hour'] = df_ab['session_start_timestamp'].dt.hour
df_ab['day_night'] = np.where((df_ab['hour'] >= 21) | (df_ab['hour'] < 6), 'Night', 'Day')

# Задание 1

In [2]:
surv_by_class = df_titanic.groupby('Pclass')['Survived'].mean()
fig1 = px.pie(values=surv_by_class.values, names=['1st', '2nd', '3rd'],
              title='1. Доля выживших по классу')
fig1.show()

In [3]:
fig2 = px.histogram(df_titanic, y='Sex', color='Survived',
                    title='2. Выживаемость по полу')
fig2.show()

In [7]:
# Преобразуем в datetime, если ещё не сделано
df_ab['session_start_timestamp'] = pd.to_datetime(df_ab['session_start_timestamp'])

# Группируем по дате и считаем среднее время до бронирования
df_ab['date'] = df_ab['session_start_timestamp'].dt.date
daily_avg_time = df_ab.groupby('date')['time_to_booking'].mean().reset_index()

fig3 = px.line(daily_avg_time, x='date', y='time_to_booking',
               title='3. Динамика времени до бронирования по дням',
               markers=True)
fig3.update_layout(xaxis_title='Дата', yaxis_title='Среднее время до бронирования (сек)')
fig3.show()

In [5]:
fig4 = px.scatter(df_ab.dropna(), x='hour', y='time_to_booking', color='booked',
                  title='4. Час суток vs время бронирования')
fig4.show()

# И снова сходятся звезды (задание *)

In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=('По полу', 'По классу', 'По возрасту',
                                    'Возраст', 'Цена билета', 'По классу и полу'))

# 1. Пол
surv_sex = pd.crosstab(df_titanic['Sex'], df_titanic['Survived'])
fig.add_trace(go.Bar(x=['Died', 'Survived'], y=surv_sex.loc['female'], name='Female'), row=1, col=1)
fig.add_trace(go.Bar(x=['Died', 'Survived'], y=surv_sex.loc['male'], name='Male'), row=1, col=1)

# 2. Класс
surv_class = df_titanic.groupby('Pclass')['Survived'].mean()
fig.add_trace(go.Bar(x=['1st', '2nd', '3rd'], y=surv_class.values), row=1, col=2)

# 3. Возраст (ребёнок/взрослый)
df_titanic['AgeGroup'] = np.where(df_titanic['Age'] <= 16, 'Child', 'Adult')
surv_age = pd.crosstab(df_titanic['AgeGroup'], df_titanic['Survived'])
fig.add_trace(go.Bar(x=['Died', 'Survived'], y=surv_age.loc['Child'], name='Child'), row=1, col=3)
fig.add_trace(go.Bar(x=['Died', 'Survived'], y=surv_age.loc['Adult'], name='Adult'), row=1, col=3)

# 4. Boxplot возраста
fig.add_trace(go.Box(x=df_titanic['Survived'].map({0:'Died',1:'Survived'}),
                     y=df_titanic['Age']), row=2, col=1)

# 5. Boxplot цены билета
fig.add_trace(go.Box(x=df_titanic['Survived'].map({0:'Died',1:'Survived'}),
                     y=df_titanic['Fare']), row=2, col=2)

# 6. Heatmap
# 6. Выжившие и погибшие по классу
surv_class_count = pd.crosstab(df_titanic['Pclass'], df_titanic['Survived'])
fig.add_trace(go.Bar(x=['1st', '2nd', '3rd'], y=surv_class_count[0], name='Погиб'), row=2, col=3)
fig.add_trace(go.Bar(x=['1st', '2nd', '3rd'], y=surv_class_count[1], name='Выжил'), row=2, col=3)

fig.update_layout(height=750, title_text="Titanic Survival Dashboard")
fig.update_xaxes(title_text="", row=1, col=1)
fig.update_xaxes(title_text="", row=1, col=2)
fig.update_xaxes(title_text="", row=1, col=3)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Survival rate", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=3)

fig.show()